# Batik Joint LoRA Trainer (SDXL) — Madura + Jawa Barat

Versi **joint training**: dua daerah batik (`madura` dan `jawa_barat`) dilatih **bersama dalam satu LoRA**, bukan dilatih terpisah lalu digabung saat inferensi seperti versi multi-region (`training2.ipynb`) sebelumnya.

**Kenapa joint training, bukan merge di inferensi?**
Merge dua LoRA yang dilatih terpisah (`set_adapters` + bobot) sering menghasilkan interferensi — motif salah satu daerah bisa "kalah"/hilang karena kedua LoRA saling menumpuk di arah bobot yang sama. Joint training menghindari masalah ini karena kedua konsep dipelajari bersama dalam satu ruang low-rank sejak awal, bukan digabung belakangan.

**Trade-off yang perlu kamu sadari:** hasil ini terikat pada pasangan `madura` + `jawa_barat` ini saja. Kalau mau ganti pasangan daerah lain, kamu perlu training ulang dari cell konfigurasi (bagian 2) — beda dengan pendekatan merge-saat-inferensi yang tinggal ganti bobot tanpa training ulang.

**Soal regularization (reg images) — tetap dipakai**, dengan skema **leave-two-out** (bukan leave-one-out seperti versi single-region). Semua daerah SELAIN `madura` dan `jawa_barat` di `ALL_REGIONS` otomatis digabung jadi reg images. Alasannya:
- **Mencegah "drift" konsep umum `batik`** — tanpa reg, kata kelas `batik` polos (tanpa trigger) bisa ikut condong hanya merepresentasikan gaya Madura+Jawa Barat saja, bukan batik secara umum.
- **Anti-overfitting** — reg images menyediakan prior preservation loss yang menahan model supaya tidak overfit ke dataset instance yang masih relatif kecil, persis fungsinya di training per-daerah biasa. Ini tidak berubah walau caption sekarang sudah menyertakan trigger token.

**Yang perlu kamu siapkan:** sama seperti notebook multi-region sebelumnya — dataset per daerah (gambar + caption `.txt` yang sudah menyertakan trigger token), ditambahkan sebagai input, GPU T4 aktif.

## 1. Clone `sd-scripts` dan install dependency
Hanya perlu dijalankan sekali di awal session.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd /kaggle/working/sd-scripts

# Cek torch bawaan Kaggle dulu (biasanya sudah CUDA-ready) sebelum install requirements,
# supaya kita tidak sengaja menimpanya dengan versi yang tidak cocok dengan GPU Kaggle.
import torch
print("Torch version (bawaan Kaggle):", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# Install requirements TANPA menimpa torch yang sudah terpasang dan sudah cocok dengan GPU Kaggle.
# Kalau baris di atas menunjukkan CUDA available: True, jalankan ini:
!grep -v -E "^diffusers\[torch\]" requirements.txt > requirements_no_torch_pin.txt
!pip install -r requirements_no_torch_pin.txt --break-system-packages -q
!pip install "diffusers==0.32.1" --no-deps --break-system-packages -q
print("Setup selesai.")


## 2. Konfigurasi 2 daerah untuk joint training & susun folder

Beda dari versi single-region: sekarang ada **`TARGET_REGIONS`** (list, bukan satu string) — isi daerah-daerah yang dilatih BERSAMA dalam satu LoRA.

`sd-scripts` mode folder klasik (tanpa `--dataset_config`) otomatis mengenali **setiap subfolder** di dalam `train_data_dir` yang mengikuti format nama `{repeat}_{trigger token} {class}` sebagai subset instance terpisah — tapi semuanya tetap dilatih ke **satu jaringan LoRA yang sama**. Makanya cukup bikin 2 subfolder (satu per daerah, masing-masing trigger token sendiri) tanpa perlu file `.toml` tambahan.

Catatan nama folder: di `ALL_REGIONS` ada `madura` dan juga `madura_pola_kompleks` / `madura_pola_sederhana` sebagai entri terpisah. Di bawah aku pakai persis nama `"madura"` dan `"jawa_barat"` sesuai permintaanmu — kalau ternyata daerah Madura yang kamu maksud itu salah satu varian pola tersebut, tinggal ganti isi `TARGET_REGIONS`.

In [ ]:
import os, shutil

# ==================== UBAH BAGIAN INI SETIAP GANTI PASANGAN DAERAH ====================
TARGET_REGIONS = ["madura", "jawa_barat"]  # daerah yang dilatih BERSAMA dalam 1 LoRA
ALL_REGIONS = [
    "madura_pola_kompleks",
    "madura_pola_sederhana",
    "jawa_barat",
    "jawa_tengah",
    "jawa_timur",
    "madura",
    "yogyakarta",
]  # SEMUA dataset batik yang kamu punya (dipakai untuk skema reg leave-two-out)
CLASS_NAME = "batik"              # kata kelas umum, dipakai di caption prior/reg
REPEATS_INSTANCE = 15             # berapa kali tiap gambar target diulang per epoch
REPEATS_REG = 1                   # berapa kali tiap gambar reg diulang per epoch
OUTPUT_NAME = "-".join(TARGET_REGIONS) + "-joint-lora"  # nama file LoRA gabungan
DATASET_ROOT = "/kaggle/input/datasets/alhamdywahyu/captioning-dataset-batik/captioned-dataset"
# ========================================================================================

base = "/kaggle/working/training-data"
img_base = f"{base}/img"
reg_dir = f"{base}/reg/{REPEATS_REG}_{CLASS_NAME}"
out_dir = f"/kaggle/working/lora-{OUTPUT_NAME}"

for d in [base, img_base, reg_dir, out_dir, f"{base}/log"]:
    os.makedirs(d, exist_ok=True)

# --- 1. Salin dataset tiap daerah TARGET ke subfolder-nya masing-masing ---
# Setiap subfolder = "{repeat}_{trigger} {class}", dikenali sd-scripts sebagai subset terpisah
# tapi tetap dilatih bersama ke satu jaringan LoRA yang sama (JOINT training).
triggers = {}
for region in TARGET_REGIONS:
    trigger = f"{region}batik"
    triggers[region] = trigger
    img_dir = f"{img_base}/{REPEATS_INSTANCE}_{trigger} {CLASS_NAME}"
    os.makedirs(img_dir, exist_ok=True)
    src = f"{DATASET_ROOT}/{region}"
    count = 0
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), img_dir)
        count += 1
    print(f"Instance images ({region}): {count} file disalin ke {img_dir}  | trigger: '{trigger}'")

# --- 2. Leave-two-out: gabungkan SEMUA daerah SELAIN kedua target sebagai reg images ---
count_reg = 0
for region in ALL_REGIONS:
    if region in TARGET_REGIONS:
        continue
    src = f"{DATASET_ROOT}/{region}"
    if not os.path.isdir(src):
        print(f"  [!] Folder {src} tidak ditemukan, dilewati.")
        continue
    for f in os.listdir(src):
        # tambahkan prefix nama daerah supaya tidak ada nama file bentrok antar dataset
        new_name = f"{region}_{f}"
        shutil.copy(os.path.join(src, f), os.path.join(reg_dir, new_name))
        count_reg += 1

print(f"\nReg images (semua daerah selain {TARGET_REGIONS}): {count_reg} file disalin ke {reg_dir}")
print(f"Trigger tokens untuk run ini: {triggers}")
print(f"Output LoRA akan bernama: {OUTPUT_NAME}.safetensors")


## 3. Training LoRA (joint: madura + jawa_barat)

Command training-nya **sama persis** dengan versi per-daerah — VAE fp16-fix tetap dipakai untuk mencegah `NaN found in latents`, `network_dim=32`/`network_alpha=16`, dst. Yang beda cuma `--train_data_dir` sekarang mengarah ke folder induk yang berisi 2 subset sekaligus (bukan 1), dan `--output_name` memakai nama gabungan.

Estimasi waktu training kira-kira 1.5–2x dari training single-region (karena total instance image kira-kira dobel), tapi tetap dibatasi `MAX_STEPS` maksimal 3000 seperti sebelumnya supaya tidak over-training.

In [ ]:
%cd /kaggle/working/sd-scripts

import os
n_instance_images = sum(
    len(os.listdir(f"{img_base}/{REPEATS_INSTANCE}_{triggers[r]} {CLASS_NAME}"))
    for r in TARGET_REGIONS
)
MAX_STEPS = min(n_instance_images * 40, 3000)
print(f"Total instance images (gabungan {len(TARGET_REGIONS)} daerah): {n_instance_images}, MAX_STEPS dipakai: {MAX_STEPS}")

!accelerate launch --num_cpu_threads_per_process=2 sdxl_train_network.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --vae="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="{img_base}" \
  --reg_data_dir="{base}/reg" \
  --output_dir="{out_dir}" \
  --output_name="{OUTPUT_NAME}" \
  --logging_dir="{base}/log" \
  --caption_extension=".txt" \
  --resolution="1024,1024" \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --text_encoder_lr=0.0004 \
  --unet_lr=0.0003 \
  --lr_scheduler="constant" \
  --lr_scheduler_num_cycles=8 \
  --train_batch_size=1 \
  --max_train_steps={MAX_STEPS} \
  --save_every_n_epochs=1 \
  --save_model_as=safetensors \
  --mixed_precision="fp16" \
  --save_precision="fp16" \
  --optimizer_type="Adafactor" \
  --optimizer_args scale_parameter=False relative_step=False warmup_init=False \
  --max_data_loader_n_workers=0 \
  --gradient_checkpointing \
  --sdpa \
  --bucket_no_upscale \
  --enable_bucket \
  --noise_offset=0.0 \
  --lowram \
  --mem_eff_attn


## 3b. Cek kurva loss training (opsional tapi disarankan)

Dijalankan tepat setelah cell training selesai, di sesi yang sama. Seharusnya TIDAK ada lagi baris "NaN found in latents" di output cell training di atas — kalau masih muncul, berarti `--vae` belum terpasang dengan benar.

In [ ]:
!pip install tensorboard --break-system-packages -q

import glob
from tensorboard.backend.event_processing import event_accumulator
import numpy as np

log_files = glob.glob(f"{base}/log/**/*tfevents*", recursive=True)
print("Event file ditemukan:", log_files)

if log_files:
    ea = event_accumulator.EventAccumulator(log_files[0])
    ea.Reload()
    print("Tag tersedia:", ea.Tags().get("scalars", []))

    loss_tag = next((t for t in ea.Tags().get("scalars", []) if "loss" in t.lower()), None)
    if loss_tag:
        events = ea.Scalars(loss_tag)
        steps = [e.step for e in events]
        vals = [e.value for e in events]
        print(f"\nTag dipakai: {loss_tag}, total {len(vals)} titik")
        print("10 nilai awal:", [round(v, 4) for v in vals[:10]])
        print("10 nilai akhir:", [round(v, 4) for v in vals[-10:]])
        print("min:", round(min(vals), 4), " max:", round(max(vals), 4))
    else:
        print("Tidak ketemu tag loss, cek nama tag di atas manual.")
else:
    print("Tidak ada event file -- pastikan cell training di atas sudah selesai jalan.")


## 4. Zip hasil LoRA gabungan

LoRA hasil joint training (`{OUTPUT_NAME}.safetensors`) sekarang **satu file saja** yang sudah memuat kedua daerah — beda dari versi merge-saat-inferensi yang menghasilkan 2 file terpisah.

Cara pakai saat generate: cukup load 1 LoRA ini dengan multiplier normal (biasanya 1.0), lalu sertakan **kedua trigger token** (`madurabatik` dan `jawa_baratbatik`) di prompt — tidak perlu lagi `region_weights` / `set_adapters` dengan bobot manual seperti pendekatan merge.

In [ ]:
!zip -j /kaggle/working/{OUTPUT_NAME}.zip {out_dir}/*.safetensors
print(f"Selesai! Download: /kaggle/working/{OUTPUT_NAME}.zip")


## Catatan

- Notebook ini **khusus untuk kombinasi tetap** `madura` + `jawa_barat`. Kalau mau coba pasangan lain (misal `madura` + `jawa_tengah`), duplikasi notebook ini dan ganti isi `TARGET_REGIONS`, lalu training ulang dari awal — joint training tidak reusable lintas kombinasi seperti pendekatan merge-saat-inferensi.
- Kalau kamu masih di tahap eksplorasi mencoba banyak pasangan berbeda, pendekatan block-wise merge (dari notebook testing terpisah, pakai LoRA per-daerah yang sudah ada) tetap lebih hemat waktu/kuota GPU untuk eksperimen cepat sebelum "mengunci" pasangan favorit lewat joint training seperti ini.
- Cek kuota GPU mingguan Kaggle sebelum training — joint training butuh waktu lebih lama dari single-region karena instance data-nya kira-kira dobel.